# SIDM ABCD signal-region datacards

Builds [Combine](https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/) datacards from
the ABCD-plane yields in the merged coffea outputs, in two flavours:

* **counting** --- one bin, the signal region (ABCD region A), background taken straight from MC;
* **ABCD** --- four bins (A, B, C, D) with the region-A background *defined* inside Combine as
  `bNorm*cNorm/dNorm`, so the control-region counts constrain it in the fit.

Then run Combine over them:

```bash
python sidm/scripts/run_combine_limits.py -j 8                       # counting
python sidm/scripts/run_combine_limits.py -j 8 \
    --datacards sidm/studies/limit_plotting/datacards_abcd \
    --pattern  'datacard_abcd_*.txt' \
    --outdir   sidm/studies/limit_plotting/limits_abcd               # ABCD
```

`limit_plots.ipynb` turns the resulting `limits.csv` into figures.

### Where the numbers come from

Each observable is a `Hist` with axes `(channel, <observable>, abcd_region)`. The ABCD plane is
built from the two lepton-jet **isolation** variables, so region A is the doubly-isolated
corner (the signal region) and D is the corner diagonally opposite --- hence the closure
relation $A = B \times C / D$.

Two things to know about the extraction, both handled by `datacard_tools.region_yields`:

* the sums use **`flow=True`**, because the observable axes are `Regular(100, 0, 700)` and
  overflow at the few-percent level. With flow included, the sum over all four regions
  reproduces the final cutflow row exactly, in both channels, for signal and background;
* the histograms are already scaled by `lumi*xs` in `sidm_processor.postprocess`, so these are
  yields, and the `Weight` storage gives each one its MC statistical error.

Signal has no entry in `cross_sections.yaml` *for the purposes of processing* --- `get_xs`
defaults to a **1 fb** reference unless `use_signal_xs=True` --- so Combine's `r` is the limit
on the signal cross section in fb. The theory cross sections are applied at plotting time in
`limit_plots.ipynb`.

In [ ]:
import sys
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(1, os.path.join(os.getcwd(), '../../..'))
sys.path.insert(1, os.getcwd())
import datacard_tools
from sidm.tools import utilities
importlib.reload(datacard_tools)

utilities.set_plot_style()
%matplotlib inline

STUDY_DIR = Path(os.getcwd())
DATACARD_DIR = STUDY_DIR / "datacards"
ABCD_DIR = STUDY_DIR / "datacards_abcd"
CACHE = STUDY_DIR / "sr_yields.pkl"

print("backgrounds:", datacard_tools.BKG_DIR)
print("signal     :", datacard_tools.SIGNAL_DIR)
for name, ch in datacard_tools.CHANNELS.items():
    print(f"{name:10s} <- {ch.selection}  (counted with {ch.hist_name})")
print("\nABCD regions:", datacard_tools.ABCD_REGIONS,
      f"  signal region = {datacard_tools.ABCD_REGIONS[datacard_tools.SR_ABCD_REGION]}")

## 0. Blinding

`datacard_tools` withholds the signal region of anything it cannot **positively identify as
simulation**. The check deliberately ignores `metadata["is_data"]`: the merge step empties that
accumulator, so it is an empty set for data and MC alike and would report real data as MC.
Instead a sample counts as simulation only if it is a known signal point or carries a cross
section in `cross_sections.yaml`; anything else is blinded.

That means the failure mode is *withholding too much*, never leaking the SR. Reading the SR of
an unrecognised sample raises `BlindingError`, and `region_yields` simply omits region A.

In [ ]:
# Demonstrate the guard on a real 2018 data file: control regions come back,
# the signal region does not.
DATA_DIR = ("/eos/uscms/store/user/dlee3/sidm_condor/ABCD_cosmic_veto/"
            "ABCD_landing_10ch_cosmic_veto_v1_data_full_merged_samples_v1")
probe = sorted(Path(DATA_DIR).glob("*.coffea"))[0]

for sample, sample_out in datacard_tools.read_coffea(probe).items():
    channel = datacard_tools.CHANNELS["SR_4mu"]
    regions = datacard_tools.region_yields(sample_out, channel, sample_name=sample)
    names = [datacard_tools.ABCD_REGIONS[r] for r in sorted(regions)]
    print(f"{sample}: is_simulation={datacard_tools.is_simulation(sample)}, "
          f"regions returned = {names}")
    try:
        datacard_tools.sr_yield(sample_out, channel, sample_name=sample)
        print("  !! the signal region was readable -- the guard is not working")
    except datacard_tools.BlindingError as err:
        print(f"  BlindingError: {err}")

## 1. Extract the ABCD-plane yields

Reading all 18 background and 120 signal files takes about a minute, so the result is cached. Set `reload = True` to re-read the coffea files.

In [ ]:
reload = False

def progress(i, n, name):
    if i % 25 == 0 or i == n - 1:
        print(f"  [{i + 1}/{n}] {name}", flush=True)

if not reload and CACHE.exists():
    cached = pd.read_pickle(CACHE)
    bkg_yields, signal_yields = cached["bkg"], cached["signal"]
    print(f"loaded cached yields from {CACHE.name}")
else:
    print("backgrounds:")
    bkg_yields = datacard_tools.collect_yields(datacard_tools.BKG_DIR, progress=progress)
    print("signal:")
    signal_yields = datacard_tools.collect_yields(datacard_tools.SIGNAL_DIR, progress=progress)
    pd.to_pickle({"bkg": bkg_yields, "signal": signal_yields}, CACHE)
    print(f"cached to {CACHE.name}")

bkg_grouped = datacard_tools.group_backgrounds(bkg_yields)
print(f"\n{len(bkg_yields)} background samples, {len(signal_yields)} signal points")

### Background in the four ABCD regions

**This is the number that matters most for reading anything below.** The control regions hold
one to seven *raw* simulated events each. Every uncertainty quoted here is MC statistical only.

In [ ]:
rows = []
for ch_name in datacard_tools.CHANNELS:
    for region in sorted(datacard_tools.ABCD_REGIONS):
        for process, y in sorted(bkg_grouped[ch_name].get(region, {}).items()):
            rows.append({"channel": ch_name, "region": datacard_tools.ABCD_REGIONS[region],
                         "process": process, "yield": y.value, "mc_stat_err": y.error,
                         "rel_err": y.rel_error,
                         "n_raw_eff": round(1 / y.rel_error**2) if y.rel_error > 0 else 0})
        total = datacard_tools.total_background(bkg_grouped, ch_name, region)
        rows.append({"channel": ch_name, "region": datacard_tools.ABCD_REGIONS[region],
                     "process": "TOTAL", "yield": total.value, "mc_stat_err": total.error,
                     "rel_err": total.rel_error,
                     "n_raw_eff": round(1 / total.rel_error**2) if total.rel_error > 0 else 0})

bkg_table = pd.DataFrame(rows)
bkg_table[bkg_table.process == "TOTAL"]

In [ ]:
# Does the ABCD closure relation hold in MC?  With this few raw events, the honest
# answer is that the test has no power -- quantify that rather than eyeball it.
for ch_name in datacard_tools.CHANNELS:
    y = {r: datacard_tools.total_background(bkg_grouped, ch_name, r)
         for r in sorted(datacard_tools.ABCD_REGIONS)}
    pred = y[1].value * y[2].value / y[3].value
    rel_pred = np.sqrt(sum(y[r].rel_error**2 for r in (1, 2, 3)))
    err = np.hypot(y[0].error, pred * rel_pred)
    print(f"{ch_name}:")
    print(f"   MC region A : {y[0].value:8.3f} +- {y[0].error:7.3f}  ({y[0].rel_error:.0%})")
    print(f"   B*C/D       : {pred:8.3f} +- {pred*rel_pred:7.3f}  ({rel_pred:.0%})")
    print(f"   difference  : {abs(y[0].value - pred)/err:.2f} sigma"
          f"  -> {'consistent; the closure test has no power here' if abs(y[0].value-pred)/err < 2 else 'possible real non-closure'}\n")

### Signal contamination of the control regions

The ABCD prediction assumes B, C and D are background-dominated. They are not: the control
regions hold so little background that even a small signal leak dominates them. Region **C** is
the problem.

This is the main reason the four-bin datacard puts signal in *all* four bins rather than only
in A --- so the fit can account for the leak instead of absorbing it into the background
prediction.

In [ ]:
rows = []
for sample, per_channel in signal_yields.items():
    info = datacard_tools.parse_signal_name(sample)
    if info is None:
        continue
    ch_name = next(n for n, c in datacard_tools.CHANNELS.items()
                   if c.signal_prefix == info["final_state"])
    regions = per_channel.get(ch_name) or {}
    row = {"signal": sample, "channel": ch_name, **info}
    for r in (1, 2, 3):
        s = regions.get(r, datacard_tools.Yield(0.0, 0.0)).value
        b = datacard_tools.total_background(bkg_grouped, ch_name, r).value
        row[f"contam_{datacard_tools.ABCD_REGIONS[r]}"] = s / (s + b) if (s + b) > 0 else 0.0
    rows.append(row)

contam = pd.DataFrame(rows)
contam["worst"] = contam[["contam_B", "contam_C", "contam_D"]].max(axis=1)
print("at the 1 fb reference (theory cross sections make it worse still):")
print(f"  >10% contamination in some control region: "
      f"{(contam.worst > 0.10).sum()}/{len(contam)} points")
contam.nlargest(10, "worst")[["signal", "channel", "contam_B", "contam_C",
                              "contam_D"]].style.format(
    {"contam_B": "{:.1%}", "contam_C": "{:.1%}", "contam_D": "{:.1%}"})

## 2. Write the datacards

Nuisance parameters:

| nuisance | applies to | model |
|---|---|---|
| `lumi_13TeV` | signal (and MC background in the counting cards) | lnN, 2.5% |
| `mcstat_<bin>_<bkg>` | each background, counting cards only | `gmN` from the raw MC entries |
| `mcstat_<bin>_signal` | signal | lnN |
| `bkg_norm` | background | off --- no non-closure systematic for now, per review |

In the **ABCD** cards the background carries no normalisation nuisance at all: `bNorm`, `cNorm`
and `dNorm` are unconstrained `rateParam`s, so the control-region counts alone determine the
background, which is the point of the method.

`target_lumi_pb` rescales both signal and background to a different luminosity. It leaves the
*relative* MC statistical errors alone, which is correct --- extrapolating to more luminosity
does not create more simulated events. Only 2018 is processed today; `run_periods.yaml` carries
commented placeholders for Run 3.

In [ ]:
config = datacard_tools.DatacardConfig(
    lumi_unc=0.025,        # 2018 integrated luminosity
    mc_stat="gmN",         # "gmN", "lnN", or None
    bkg_norm_unc=None,     # no non-closure systematic for now
    signal_unc=None,       # signal systematics still to come
    target_lumi_pb=None,   # e.g. 4*59830 to extrapolate; None = leave at 2018
    observation="bkg",     # blinded
)

counting, floored = datacard_tools.write_datacards(
    signal_yields, bkg_grouped, DATACARD_DIR, config=config)
print(f"wrote {len(counting)} counting datacards to {DATACARD_DIR}")
if floored:
    print(f"  WARNING: {len(floored)} had no non-empty background and were floored")

abcd, warnings = datacard_tools.write_abcd_datacards(
    signal_yields, bkg_grouped, ABCD_DIR, config=config)
print(f"wrote {len(abcd)} ABCD datacards to {ABCD_DIR}")
for w in warnings[:5]:
    print(f"  WARNING: {w}")

In [ ]:
example = ABCD_DIR / "datacard_abcd_SR_4mu_4Mu_800GeV_5p0GeV_5p0mm.txt"
print(example.read_text())

## 3. Run Combine

Combine lives in its own CMSSW release, so run the script from a shell rather than importing it:

```bash
python sidm/scripts/run_combine_limits.py -j 8
python sidm/scripts/run_combine_limits.py -j 8 \
    --datacards sidm/studies/limit_plotting/datacards_abcd \
    --pattern  'datacard_abcd_*.txt' \
    --outdir   sidm/studies/limit_plotting/limits_abcd
```

The script picks Combine's `--rMax` per card from that card's S and B, because the faintest
signal points sit at `r` of order 1e6 and fall outside Combine's default range of 20. For the
four-bin cards it reads S and B from the `_A` bin specifically, since the ABCD background rate
column is 1 by construction.

In [ ]:
import subprocess

for extra in ([], ["--datacards", str(ABCD_DIR), "--pattern", "datacard_abcd_*.txt",
                   "--outdir", str(STUDY_DIR / "limits_abcd")]):
    result = subprocess.run(
        [sys.executable, "../../scripts/run_combine_limits.py", "-j", "8"] + extra,
        capture_output=True, text=True)
    print(result.stdout.strip().splitlines()[-1] if result.stdout else result.stderr[-500:])

## 4. Counting vs ABCD

How much the background estimate changes the answer.

In [ ]:
counting_csv = STUDY_DIR / "limits" / "limits.csv"
abcd_csv = STUDY_DIR / "limits_abcd" / "limits.csv"
if counting_csv.exists() and abcd_csv.exists():
    c = pd.read_csv(counting_csv)
    a = pd.read_csv(abcd_csv)
    merged = c.merge(a, on=["channel", "signal"], suffixes=("_counting", "_abcd"))
    merged["ratio"] = merged.exp_abcd / merged.exp_counting
    for ch, group in merged.groupby("channel"):
        y = {r: datacard_tools.total_background(bkg_grouped, ch, r)
             for r in sorted(datacard_tools.ABCD_REGIONS)}
        pred = y[1].value * y[2].value / y[3].value
        print(f"{ch}: ABCD/counting limit ratio = {group.ratio.median():.3f} "
              f"(MC A = {y[0].value:.3f}, B*C/D = {pred:.3f})")
    print("\nThe ABCD limits are stronger only because B*C/D happens to sit below the MC")
    print("region-A count. That difference is under 1 sigma given the MC statistics, so it")
    print("is a fluctuation, not a real gain in sensitivity.")